# Data Collection
## Pulling and caching market data with yfinance

The first stage of any financial TDA pipeline is reliable data collection. We pull historical OHLCV data for three assets across different asset classes and cache it locally for efficient reuse across all downstream repos.

We work with three tickers to cover different asset classes and stress-test our pipeline:

- **`^GSPC`** — S&P 500 index. Clean, complete data from 2000. Benchmark.
- **`TSLA`** — Tesla Inc. High volatility stock, post-2010 only. We artificially introduce missing data to simulate trading halts and data provider errors.
- **`BTC-USD`** — Bitcoin. Crypto asset, post-2014. Interesting behavior during 2020 crisis — crashed with markets then decoupled.

In [ ]:
import sys
sys.path.insert(0, '/home/gabo-linux/TDA-Gabo/tda-financial-data-pipeline')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data import fetch_ticker

In [ ]:
# Pull all three tickers
sp500 = fetch_ticker('^GSPC', '2000-01-01', '2024-01-01')
tsla = fetch_ticker('TSLA', '2000-01-01', '2024-01-01')
btc = fetch_ticker('BTC-USD', '2000-01-01', '2024-01-01')

# Summary
print("Asset coverage summary:")
print(f"{'Ticker':<10} {'Start':<25} {'End':<25} {'Days':<10}")
print("-" * 70)
for name, df in [('S&P 500', sp500), ('TSLA', tsla), ('BTC-USD', btc)]:
    print(f"{name:<10} {str(df.index[0]):<25} {str(df.index[-1]):<25} {len(df):<10}")

## Asset Coverage

| Ticker | Start | Days | Asset Class |
|--------|-------|------|-------------|
| S&P 500 | Jan 2000 | 6037 | Equity Index |
| TSLA | Jun 2010 | 3400 | Individual Stock |
| BTC-USD | Sep 2014 | 3393 | Cryptocurrency |

Three different asset classes, three different date ranges. The pipeline must handle each independently and align them correctly for joint analysis.

Note the timezone differences — S&P 500 and TSLA are in US Eastern time, BTC trades 24/7 in UTC. This will require careful alignment in preprocessing.

## Observations

- S&P 500 has 6037 trading days starting January 2000
- TSLA has 3400 trading days starting June 29 2010 — its IPO date
- Aligning both series requires handling 2637 missing days at the start

This is a common real-world problem in quantitative finance — assets have different trading histories and any pipeline must handle misaligned time series gracefully.

## Timezone Alignment

S&P 500 and TSLA data is in US Eastern time, BTC is in UTC. Mixed timezone-aware indices cause rendering issues and complicate date arithmetic. We convert all indices to timezone-naive UTC before proceeding.

In [ ]:
sp500.index = pd.to_datetime(sp500.index, utc=True).tz_localize(None)
tsla.index = pd.to_datetime(tsla.index, utc=True).tz_localize(None)
btc.index = pd.to_datetime(btc.index, utc=True).tz_localize(None)

## Visual Overview
### Close price history for each asset

A first look at the raw price series — different scales, different histories, different behaviors. Notice how each asset reflects its own market dynamics.

In [ ]:
import matplotlib
matplotlib.rcParams.update(matplotlib.rcParamsDefault)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), facecolor='white')

assets = [
    (sp500, 'S&P 500 (^GSPC)', 'steelblue'),
    (tsla, 'Tesla (TSLA)', 'darkorange'),
    (btc, 'Bitcoin (BTC-USD)', 'green')
]

for ax, (df, title, color) in zip(axes, assets):
    ax.set_facecolor('white')
    ax.plot(df.index, df['Close'], color=color, linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel('Price (USD)')
    ax.grid(alpha=0.3, color='lightgrey')

plt.tight_layout()
plt.show()

## Price History with Crisis Periods

All three assets plotted on a shared time axis — allowing direct visual comparison of how each reacted to the three major market crises. Assets that didn't exist during a crisis simply show no data for that period.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), facecolor='white', sharex=True)

assets = [
    (sp500, 'S&P 500 (^GSPC)', 'steelblue'),
    (tsla, 'Tesla (TSLA)', 'darkorange'),
    (btc, 'Bitcoin (BTC-USD)', 'green')
]

crises = [
    ('2000-03-01', '2002-10-01', 'red', 'Dot-com'),
    ('2008-09-01', '2009-06-01', 'orange', 'GFC'),
    ('2020-02-01', '2020-04-01', 'purple', 'COVID')
]

for ax, (df, title, color) in zip(axes, assets):
    ax.set_facecolor('white')
    ax.plot(df.index, df['Close'], color=color, linewidth=0.8)
    for start, end, c, label in crises:
        ax.axvspan(start, end, alpha=0.15, color=c, label=label)
    ax.set_title(title)
    ax.set_ylabel('Price (USD)')
    ax.grid(alpha=0.3, color='lightgrey')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## Observations

All three assets show a clear reaction to the 2020 COVID crash:

- **S&P 500** — sharp drawdown followed by steady recovery
- **TSLA** — dropped with the market then dramatically outperformed during recovery
- **BTC** — crashed alongside traditional markets in March 2020, then decoupled with an extraordinary rally

This raises a key TDA question for later repos: **did these assets share topological structure during the crash, and when did they diverge?**

## Price Series

The three crisis periods are visible across all assets — though each reacts differently based on its nature and history. Our TDA pipeline will operate on **returns**, not prices — so the next step is preprocessing.

Note: crisis windows are approximate and used for visual reference only. Assets that did not exist during a crisis show no data for that period.

## Reusable Data Collection

The data collection logic is encapsulated in `src/data.py` — making it reusable across all future repos. Any ticker, any date range, with automatic local caching.

Used here for three assets: S&P 500, TSLA, and BTC-USD. The same function will be used in all downstream repos without modification.

## Conclusions

Data collection pipeline complete:

- Three assets pulled and cached locally via `fetch_ticker()`
- Timezone alignment applied — all indices converted to timezone-naive UTC
- Different asset classes, date ranges and market behaviors captured
- TSLA will have artificial missing data introduced in Notebook 02 to simulate real-world pipeline conditions

All raw data cached in `data/raw/` — subsequent runs load instantly without API calls.